# 2e — Koszul bracket anti-symmetry: $[\alpha, \beta]_{T^*M} = -[\beta, \alpha]_{T^*M}$

**Problem (e).** $T^*M$ üzerinde Koszul bracket anti-simetrik:

$$
[\alpha, \beta]_{T^*M} = -[\beta, \alpha]_{T^*M}.
$$

Koszul bracket'in tanımı:

$$
[\alpha, \beta]_K \;=\; \mathcal{L}_{\pi^\sharp\alpha}\beta \;-\; \mathcal{L}_{\pi^\sharp\beta}\alpha \;-\; d\langle \pi^\sharp\alpha, \beta \rangle.
$$

İki Koszul tanımını topladığımızda Lie-türev terimleri kendi karşıtlarıyla götürülür; geriye yalnızca $-d\bigl(\langle X_\alpha,\beta\rangle + \langle X_\beta,\alpha\rangle\bigr)$ kalır. $\pi$ anti-simetrik olduğu için bu pairing toplamı sıfırdır.

## Strateji — niye 2d'den belirgin daha kolay

2d **$C^\infty$-modül cebri** istiyordu: sharp tensoriyalliği, $\mathcal{L}_{fX}$ açılımı, $df\wedge\iota$. 2e'de bunların **hiçbiri** yok — anti-symmetry saf cebirsel bir iptal:

$$
[\alpha,\beta]_K + [\beta,\alpha]_K \;=\; \bigl(L_{X_\alpha}\beta - L_{X_\beta}\alpha\bigr) + \bigl(L_{X_\beta}\alpha - L_{X_\alpha}\beta\bigr) - d\langle X_\alpha,\beta\rangle - d\langle X_\beta,\alpha\rangle.
$$

Lie-türev kısmı **doğrudan** birbirini götürüyor — açılmaya gerek yok. Geriye:

$$
-d\bigl(\langle X_\alpha,\beta\rangle + \langle X_\beta,\alpha\rangle\bigr) \;=\; -d(\pi(\alpha,\beta) + \pi(\beta,\alpha)) \;=\; 0.
$$

**Tek geometrik girdi:** $\pi$ anti-simetri, pairing seviyesinde: $\langle X_\alpha,\beta\rangle = -\langle X_\beta,\alpha\rangle$.

**Mode disiplini.** Her iki vektör alanı **flow-mode**. Cartan-mode olsalardı $L_{X_\alpha}\beta \to (d\iota_{X_\alpha} + \iota_{X_\alpha}d)\beta$ açılır, $d(\beta)$ generic 1-formdan başka bir şeye indirgenemez ve zincir tıkanırdı. Cancel olmaları için **opak kalmaları** lazım — `flow` tam olarak bunu sağlıyor.

In [1]:
# Notebook doğrudan açıldığında jacopy'ı import edilebilir hâle getirir.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

from jacopy.algebra.derivation import Act, Derivation
from jacopy.calculus.exterior_d import d
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.calculus.pairing import pairing, Pairing
from jacopy.core.expr import Neg, Sum, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import Definition, default_engine
from jacopy.proof.strategies import ExpandAndSimplify

## 1. Kurulum

- $\alpha, \beta$ — generic 1-formlar.
- $X_\alpha = \pi^\sharp(\alpha)$, $X_\beta = \pi^\sharp(\beta)$ — isimli `Derivation`'lar (sharp'ı bir tensör olarak intrinsik tutmak Faz 12 #6'ya bağlı; bu pass'te ürün vektör alanlarını doğrudan model'liyoruz).
- $\mathcal{L}_{X_\alpha}, \mathcal{L}_{X_\beta}$ **flow-mode** — Lie-türev terimleri opak kalıp birbirini götürsün diye.

In [2]:
reg = PropertyRegistry()

alpha = Symbol("α")
beta = Symbol("β")
reg.declare(alpha, Graded(degree=1))
reg.declare(beta, Graded(degree=1))

X_a = Derivation("X_α", degree=0)
X_b = Derivation("X_β", degree=0)
L_Xa = lie_derivative(X_a, definition="flow")
L_Xb = lie_derivative(X_b, definition="flow")

koszul_ab = Symbol("[α,β]_K")
koszul_ba = Symbol("[β,α]_K")
reg.declare(koszul_ab, Graded(degree=1))
reg.declare(koszul_ba, Graded(degree=1))

print(f"α, β       : {alpha} {beta}")
print(f"X_α, X_β   : {X_a} {X_b}  (her ikisi flow-mode)")
print(f"[α,β]_K    : {koszul_ab}")
print(f"[β,α]_K    : {koszul_ba}")

α, β       : α β
X_α, X_β   : X_α X_β  (her ikisi flow-mode)
[α,β]_K    : [α,β]_K
[β,α]_K    : [β,α]_K


## 2. Aksiyomlar (3 tane)

**A-K1 — `[α, β]_K` defining.** Standart Koszul tanımı:

$$
[\alpha, \beta]_K \;\to\; \mathcal{L}_{X_\alpha}\beta - \mathcal{L}_{X_\beta}\alpha - d\langle X_\alpha, \beta\rangle.
$$

**A-K2 — `[β, α]_K` defining.** Aynı formül, rolleri değiş:

$$
[\beta, \alpha]_K \;\to\; \mathcal{L}_{X_\beta}\alpha - \mathcal{L}_{X_\alpha}\beta - d\langle X_\beta, \alpha\rangle.
$$

**A-π-pairing-antisym — pairing seviyesinde $\pi$ anti-simetri.**

$$
\langle X_\alpha, \beta\rangle = -\langle X_\beta, \alpha\rangle.
$$

Çünkü $\langle\pi^\sharp\alpha,\beta\rangle = \beta(\pi^\sharp\alpha) = \pi(\alpha,\beta) = -\pi(\beta,\alpha) = -\langle\pi^\sharp\beta,\alpha\rangle$. Tek **gerçek geometrik girdi** — Faz 12 altyapı #11 (`AntiSymmetric` registry property + bivector eval rewrite) landing ettiğinde aksiyomdan theorem'e iner.

In [3]:
class KoszulDef_ab(Definition):
    """A-K1: [α, β]_K = L_Xα(β) − L_Xβ(α) − d⟨X_α, β⟩."""
    name = "[α,β]_K = L_Xα(β) − L_Xβ(α) − d⟨X_α, β⟩"
    def matches(self, expr): return expr == koszul_ab
    def rewrite(self, expr):
        return Sum(
            Act(L_Xa, beta),
            Neg(Act(L_Xb, alpha)),
            Neg(Act(d, pairing(X_a, beta))),
        )


class KoszulDef_ba(Definition):
    """A-K2: [β, α]_K = L_Xβ(α) − L_Xα(β) − d⟨X_β, α⟩."""
    name = "[β,α]_K = L_Xβ(α) − L_Xα(β) − d⟨X_β, α⟩"
    def matches(self, expr): return expr == koszul_ba
    def rewrite(self, expr):
        return Sum(
            Act(L_Xb, alpha),
            Neg(Act(L_Xa, beta)),
            Neg(Act(d, pairing(X_b, alpha))),
        )


class PairingAntisym(Definition):
    """A-π-pairing-antisym: ⟨X_α, β⟩ = −⟨X_β, α⟩."""
    name = "⟨X_α, β⟩ = −⟨X_β, α⟩ (π anti-symmetry)"
    def matches(self, expr):
        return (isinstance(expr, Pairing) and expr.alpha == X_a and expr.X == beta)
    def rewrite(self, expr):
        return Neg(pairing(X_b, alpha))


print("- [α,β]_K = L_Xα(β) − L_Xβ(α) − d⟨X_α, β⟩")
print("- [β,α]_K = L_Xβ(α) − L_Xα(β) − d⟨X_β, α⟩")
print("- ⟨X_α, β⟩ = −⟨X_β, α⟩  (π anti-symmetry)")

- [α,β]_K = L_Xα(β) − L_Xβ(α) − d⟨X_α, β⟩
- [β,α]_K = L_Xβ(α) − L_Xα(β) − d⟨X_β, α⟩
- ⟨X_α, β⟩ = −⟨X_β, α⟩  (π anti-symmetry)


## 3. Engine

`default_engine` $d^2=0$, pairing kuralları, product-rule canonical form'u zaten taşır. Üzerine üç problem-aksiyomunu ekliyoruz.

In [4]:
engine = default_engine(registry=reg, d_squared_mode="axiom")
engine.register(KoszulDef_ab())
engine.register(KoszulDef_ba())
engine.register(PairingAntisym())

print(f"engine carries {len(engine.definitions)} definitions")
for defn in engine.definitions:
    print(" -", defn.name)

engine carries 11 definitions
 - L_X := d∘ι_X + ι_X∘d (Cartan definition)
 - L_X(f) = X(f) on 0-forms (flow)
 - L_X ∘ d = d ∘ L_X (flow)
 - Act linearity: (A + B)(x) = A(x) + B(x)
 - d² = 0
 - ι_X ∘ ι_X = 0
 - ι_X(f) = 0 on 0-forms
 - ι_X(df) = X(f)
 - [α,β]_K = L_Xα(β) − L_Xβ(α) − d⟨X_α, β⟩
 - [β,α]_K = L_Xβ(α) − L_Xα(β) − d⟨X_β, α⟩
 - ⟨X_α, β⟩ = −⟨X_β, α⟩ (π anti-symmetry)


## 4. Hedef ve ispat

$$
[\alpha, \beta]_K \;=\; -[\beta, \alpha]_K.
$$

In [5]:
lhs = koszul_ab
rhs = Neg(koszul_ba)

print(f"LHS: {lhs}")
print(f"RHS: {rhs}")

chain = ExpandAndSimplify().prove(lhs, rhs, registry=reg, engine=engine)
print(f"\nKAPANDI — {len(chain)} adım.")

LHS: [α,β]_K
RHS: (-[β,α]_K)

KAPANDI — 5 adım.


## 5. İspat zinciri — LaTeX

In [6]:
from jacopy.display.jupyter import display_chain
display_chain(chain)

\begin{align*}
[\alpha,\beta]_K &\to L_{X_}\alpha\!\left(\beta\right) - L_{X_}\beta\!\left(\alpha\right) - d\!\left(\langle X_\alpha,\, \beta \rangle\right) && \text{[[\ensuremath{\alpha},\ensuremath{\beta}]\_K = L\_X\ensuremath{\alpha}(\ensuremath{\beta}) − L\_X\ensuremath{\beta}(\ensuremath{\alpha}) − d\ensuremath{\langle}X\_\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle}]\,(axiom)}\;\text{--- apply axiom: [\ensuremath{\alpha},\ensuremath{\beta}]\_K = L\_X\ensuremath{\alpha}(\ensuremath{\beta}) − L\_X\ensuremath{\beta}(\ensuremath{\alpha}) − d\ensuremath{\langle}X\_\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle}} \\
\langle X_\alpha,\, \beta \rangle &\to -\langle X_\beta,\, \alpha \rangle && \text{[\ensuremath{\langle}X\_\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} = −\ensuremath{\langle}X\_\ensuremath{\beta}, \ensuremath{\alpha}\ensuremath{\rangle} (\ensuremath{\pi} anti-symmetry)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\langle}X\_\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} = −\ensuremath{\langle}X\_\ensuremath{\beta}, \ensuremath{\alpha}\ensuremath{\rangle} (\ensuremath{\pi} anti-symmetry)} \\
[\beta,\alpha]_K &\to L_{X_}\beta\!\left(\alpha\right) - L_{X_}\alpha\!\left(\beta\right) - d\!\left(\langle X_\beta,\, \alpha \rangle\right) && \text{[[\ensuremath{\beta},\ensuremath{\alpha}]\_K = L\_X\ensuremath{\beta}(\ensuremath{\alpha}) − L\_X\ensuremath{\alpha}(\ensuremath{\beta}) − d\ensuremath{\langle}X\_\ensuremath{\beta}, \ensuremath{\alpha}\ensuremath{\rangle}]\,(axiom)}\;\text{--- apply axiom: [\ensuremath{\beta},\ensuremath{\alpha}]\_K = L\_X\ensuremath{\beta}(\ensuremath{\alpha}) − L\_X\ensuremath{\alpha}(\ensuremath{\beta}) − d\ensuremath{\langle}X\_\ensuremath{\beta}, \ensuremath{\alpha}\ensuremath{\rangle}} \\
\left(L_{X_}\alpha\!\left(\beta\right) - L_{X_}\beta\!\left(\alpha\right) - d\!\left(-\langle X_\beta,\, \alpha \rangle\right)\right) - \left(-\left(L_{X_}\beta\!\left(\alpha\right) - L_{X_}\alpha\!\left(\beta\right) - d\!\left(\langle X_\beta,\, \alpha \rangle\right)\right)\right) &\to \left(L_{X_}\alpha\!\left(\beta\right) - L_{X_}\beta\!\left(\alpha\right) - \left(-d\!\left(\langle X_\beta,\, \alpha \rangle\right)\right)\right) - \left(-\left(L_{X_}\beta\!\left(\alpha\right) - L_{X_}\alpha\!\left(\beta\right) - d\!\left(\langle X_\beta,\, \alpha \rangle\right)\right)\right) && \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
\left(L_{X_}\alpha\!\left(\beta\right) - L_{X_}\beta\!\left(\alpha\right) - \left(-d\!\left(\langle X_\beta,\, \alpha \rangle\right)\right)\right) - \left(-\left(L_{X_}\beta\!\left(\alpha\right) - L_{X_}\alpha\!\left(\beta\right) - d\!\left(\langle X_\beta,\, \alpha \rangle\right)\right)\right) &\to 0 && \text{[simplify]}\;\text{--- canonical-form pipeline}
\end{align*}

## 6. Adım adım

Zincirin akışı (5 adım):

1. **A-K1** — LHS açılımı: $[\alpha,\beta]_K \to L_{X_\alpha}\beta - L_{X_\beta}\alpha - d\langle X_\alpha,\beta\rangle$.
2. **A-π-pairing-antisym** — $\langle X_\alpha,\beta\rangle \to -\langle X_\beta,\alpha\rangle$. Tek geometrik adım.
3. **A-K2** — RHS'taki $[\beta,\alpha]_K$ açılımı (negation altında).
4. **product-rule** — `Neg`'lerin canonical-form pipeline'ı: çift negation iptali, `Sum` distribute.
5. **simplify** — kanonik form: tüm $L_X$ terimleri ve pairing $d$'leri eşleşip $0$'a düşer.

In [7]:
for i, step in enumerate(chain.steps, 1):
    print(f"[{i}] {step.rule}")
    print(f"    {step.before}")
    print(f" ↦  {step.after}")
    print()

[1] [α,β]_K = L_Xα(β) − L_Xβ(α) − d⟨X_α, β⟩
    [α,β]_K
 ↦  (L_X_α(β) + (-L_X_β(α)) + (-d(⟨X_α, β⟩)))

[2] ⟨X_α, β⟩ = −⟨X_β, α⟩ (π anti-symmetry)
    ⟨X_α, β⟩
 ↦  (-⟨X_β, α⟩)

[3] [β,α]_K = L_Xβ(α) − L_Xα(β) − d⟨X_β, α⟩
    [β,α]_K
 ↦  (L_X_β(α) + (-L_X_α(β)) + (-d(⟨X_β, α⟩)))

[4] product-rule
    ((L_X_α(β) + (-L_X_β(α)) + (-d((-⟨X_β, α⟩)))) + (-(-(L_X_β(α) + (-L_X_α(β)) + (-d(⟨X_β, α⟩))))))
 ↦  ((L_X_α(β) + (-L_X_β(α)) + (-(-d(⟨X_β, α⟩)))) + (-(-(L_X_β(α) + (-L_X_α(β)) + (-d(⟨X_β, α⟩))))))

[5] simplify
    ((L_X_α(β) + (-L_X_β(α)) + (-(-d(⟨X_β, α⟩)))) + (-(-(L_X_β(α) + (-L_X_α(β)) + (-d(⟨X_β, α⟩))))))
 ↦  0



## Sonuç

$$
\boxed{\;[\alpha, \beta]_{T^*M} = -[\beta, \alpha]_{T^*M}.\;}
$$

**5-adımlı zincir, 3 inline aksiyom.** 2d ile karşılaştırma:

| Pass | Adım | Aksiyom | Geometrik içerik |
|---|---|---|---|
| 2d (Leibniz) | 13 | 4 | sharp linearity + $L_{fX}$ + pairing linearity + π anti-sym |
| 2e (anti-sym) | 5 | 3 | yalnızca π anti-sym |

**Faz 12 etiketleri.** İdealde:

- **Altyapı #11** — `AntiSymmetric` registry property + bivector evaluation rewrite ($\pi(\alpha,\beta)\to-\pi(\beta,\alpha)$): A-π-pairing-antisym aksiyomdan theorem'e iner.
- **12.C(f) `KoszulProblem(π, forms, engine)` wrapper**: A-K1 + A-K2'yi otomatik kaydeder.

Bu ikisi birlikte landing ettiğinde 2e callsite'ı **sıfır aksiyoma** iner — saf engine zinciri kalır.